In [0]:
dbutils.widgets.dropdown("use_unity_catalog", "true", ["true", "false"], "Use Unity Catalog")
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_prefix", "retail", "Schema Prefix")
dbutils.widgets.text("landing_path", "/Volumes/workspace/default/retail_landing", "Landing Path")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("task_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
    StructField("rows_affected", LongType(), True),
    StructField("log_timestamp", StringType(), True),
])

def log_event(layer, task_name, status, message="", rows_affected=None):
    row = [(
        RUN_ID, layer, task_name, status, str(message)[:2000],
        int(rows_affected) if rows_affected is not None else None,
        time.strftime("%Y-%m-%d %H:%M:%S")
    )]
    log_df = (spark.createDataFrame(row, schema=LOG_SCHEMA)
              .withColumn("log_timestamp", col("log_timestamp").cast("timestamp")))
    if not spark.catalog.tableExists(LOG_TABLE):
        log_df.write.format("delta").mode("overwrite").saveAsTable(LOG_TABLE)
    else:
        log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)
    print(f"[{status}] ({layer}) {task_name}: {message}")
    

In [0]:
%run ./00b_pipeline_utils

In [0]:
datasets = [
    {"prefix": "olist_customers_dataset", "table": "raw_customers"},
    {"prefix": "olist_geolocation_dataset", "table": "raw_geolocation"},
    {"prefix": "olist_order_items_dataset", "table": "raw_order_items"},
    {"prefix": "olist_order_payments_dataset", "table": "raw_order_payments"},
    {"prefix": "olist_order_reviews_dataset", "table": "raw_order_reviews"},
    {"prefix": "olist_orders_dataset", "table": "raw_orders"},
    {"prefix": "olist_products_dataset", "table": "raw_products"},
    {"prefix": "olist_sellers_dataset", "table": "raw_sellers"},
    {"prefix": "product_category_name_translation", "table": "raw_product_category_name_translation"},
]

In [0]:
for d in datasets:
    spark.sql(f"DROP TABLE IF EXISTS {bronze_db}.{d['table']}")
print("Old bronze tables cleared — Auto Loader will rebuild them fresh.")

Old bronze tables cleared — Auto Loader will rebuild them fresh.


In [0]:
from pyspark.sql.functions import current_timestamp, col

def ingest_to_bronze(prefix, target_table):
    schema_loc = f"{landing_path}/_checkpoints/{target_table}/schema"
    checkpoint_loc = f"{landing_path}/_checkpoints/{target_table}/checkpoint"

    stream_df = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "csv")
                 .option("cloudFiles.schemaLocation", schema_loc)
                 .option("header", "true")
                 .option("multiLine", "true")
                 .option("quote", '"')
                 .option("escape", '"')
                 .option("cloudFiles.inferColumnTypes", "true")
                 .option("pathGlobFilter", f"{prefix}*.csv")
                 .load(landing_path))

    processed_df = (stream_df
                    .withColumn("_ingestion_timestamp", current_timestamp())
                    .withColumn("_source_file_name", col("_metadata.file_path")))

    query = (processed_df.writeStream
             .format("delta")
             .option("checkpointLocation", checkpoint_loc)
             .option("mergeSchema", "true")
             .outputMode("append")
             .trigger(availableNow=True)
             .toTable(f"{bronze_db}.{target_table}"))

    query.awaitTermination()
    return spark.table(f"{bronze_db}.{target_table}").count()

In [0]:
succeeded, failed = [], []

for d in datasets:
    log_event("bronze", d["table"], "START", f"Auto Loader scanning for {d['prefix']}*.csv")
    try:
        row_count = ingest_to_bronze(d["prefix"], d["table"])
        log_event("bronze", d["table"], "SUCCESS", rows_affected=row_count)
        succeeded.append(d["table"])
    except Exception as e:
        log_event("bronze", d["table"], "FAIL", message=str(e))
        failed.append(d["table"])

print(f"\nSucceeded: {succeeded}")
print(f"Failed: {failed if failed else 'none'}")
if len(failed) == len(datasets):
    raise Exception("Bronze ingestion failed for ALL datasets.")

[START] (bronze) raw_customers: Auto Loader scanning for olist_customers_dataset*.csv
[SUCCESS] (bronze) raw_customers: 
[START] (bronze) raw_geolocation: Auto Loader scanning for olist_geolocation_dataset*.csv
[SUCCESS] (bronze) raw_geolocation: 
[START] (bronze) raw_order_items: Auto Loader scanning for olist_order_items_dataset*.csv
[SUCCESS] (bronze) raw_order_items: 
[START] (bronze) raw_order_payments: Auto Loader scanning for olist_order_payments_dataset*.csv
[SUCCESS] (bronze) raw_order_payments: 
[START] (bronze) raw_order_reviews: Auto Loader scanning for olist_order_reviews_dataset*.csv
[SUCCESS] (bronze) raw_order_reviews: 
[START] (bronze) raw_orders: Auto Loader scanning for olist_orders_dataset*.csv
[SUCCESS] (bronze) raw_orders: 
[START] (bronze) raw_products: Auto Loader scanning for olist_products_dataset*.csv
[SUCCESS] (bronze) raw_products: 
[START] (bronze) raw_sellers: Auto Loader scanning for olist_sellers_dataset*.csv
[SUCCESS] (bronze) raw_sellers: 
[START] (br

In [0]:
display(spark.sql(f"SHOW TABLES IN {bronze_db}"))

database,tableName,isTemporary
retail_bronze,raw_customers,false
retail_bronze,raw_geolocation,false
retail_bronze,raw_order_items,false
retail_bronze,raw_order_payments,false
retail_bronze,raw_order_reviews,false
retail_bronze,raw_orders,false
retail_bronze,raw_product_category_name_translation,false
retail_bronze,raw_products,false
retail_bronze,raw_sellers,false


In [0]:
# Step 1: drop the broken bronze table
spark.sql(f"DROP TABLE IF EXISTS {bronze_db}.raw_order_reviews")

# Step 2: clear its checkpoint so Auto Loader re-reads the file instead of skipping it
checkpoint_path = f"{landing_path}/_checkpoints/raw_order_reviews"
dbutils.fs.rm(checkpoint_path, recurse=True)
print("Cleared old table and checkpoint for raw_order_reviews")

# Step 3: re-ingest with proper multi-line CSV handling
from pyspark.sql.functions import current_timestamp, col

stream_df = (spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "csv")
             .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
             .option("header", "true")
             .option("cloudFiles.inferColumnTypes", "true")
             .option("multiLine", "true")
             .option("quote", '"')
             .option("escape", '"')
             .option("pathGlobFilter", "olist_order_reviews_dataset*.csv")
             .load(landing_path))

processed_df = (stream_df
                .withColumn("_ingestion_timestamp", current_timestamp())
                .withColumn("_source_file_name", col("_metadata.file_path")))

query = (processed_df.writeStream
         .format("delta")
         .option("checkpointLocation", f"{checkpoint_path}/checkpoint")
         .option("mergeSchema", "true")
         .outputMode("append")
         .trigger(availableNow=True)
         .toTable(f"{bronze_db}.raw_order_reviews"))

query.awaitTermination()
print("Fixed row count:", spark.table(f"{bronze_db}.raw_order_reviews").count())

Cleared old table and checkpoint for raw_order_reviews
Fixed row count: 99224
